In [2]:
import importlib
import os
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

import dtree_process_worker as _dtree_worker

# Force refresh in notebooks so newly added helpers are visible.
_dtree_worker = importlib.reload(_dtree_worker)

if hasattr(_dtree_worker, "build_class_folds_worker"):
    build_class_folds_worker = _dtree_worker.build_class_folds_worker
else:
    # Fallback keeps notebook runnable even if module is stale.
    def build_class_folds_worker(args):
        class_value, class_indices, k, seed = args
        indices = np.asarray(class_indices, dtype=np.int64).copy()
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)
        split_indices = np.array_split(indices, int(k))
#         return class_value, [chunk.astype(np.int64) for chunk in split_indices]


In [5]:
def entropy_from_labels(labels):
    labels_arr = np.asarray(labels, dtype=np.int64)
    if labels_arr.size == 0:
        return 0.0
    _, counts = np.unique(labels_arr, return_counts=True)
    probs = counts / counts.sum()
    return float(-np.sum(probs * np.log2(probs)))


def majority_label(labels):
    labels_arr = np.asarray(labels, dtype=np.int64)
    values, counts = np.unique(labels_arr, return_counts=True)
    return int(values[np.argmax(counts)])


def split_dataset_np(dataset, feature_index, feature_value):
    data = np.asarray(dataset)
    if data.size == 0:
        return data
    mask = data[:, feature_index] == feature_value
    filtered = data[mask]
    if filtered.size == 0:
        return np.empty((0, data.shape[1] - 1), dtype=data.dtype)
    return np.delete(filtered, feature_index, axis=1)


def _feature_gain_worker(args):
    dataset, feature_index, base_entropy = args
    data = np.asarray(dataset)
    values = np.unique(data[:, feature_index])
    total = float(data.shape[0])
    conditional_entropy = 0.0

    for value in values:
        subset = split_dataset_np(data, feature_index, value)
        if subset.shape[0] == 0:
            continue
        conditional_entropy += (subset.shape[0] / total) * entropy_from_labels(subset[:, -1])

    return feature_index, base_entropy - conditional_entropy


def best_feature_threaded(dataset, thread_workers=8):
    data = np.asarray(dataset)
    n_features = data.shape[1] - 1
    if n_features <= 1:
        return 0

    base_entropy = entropy_from_labels(data[:, -1])
    args = np.empty(n_features, dtype=object)
    for idx in range(n_features):
        args[idx] = (data, idx, base_entropy)

    workers = max(1, min(int(thread_workers), n_features))
    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = tuple(executor.map(_feature_gain_worker, args))

    return int(max(results, key=lambda item: item[1])[0])


def build_tree(dataset, feature_names, min_sample_size=1, max_depth=None, depth=0, thread_workers=8):
    data = np.asarray(dataset)
    names = np.asarray(feature_names, dtype=object)

    labels = data[:, -1]

    # stop condition1: when number of samples less then setted
    if np.unique(labels).size <= max(1, min_sample_size):
        return majority_label(labels)

    # stop condition2: run out of features
    if data.shape[1] == 1:
        return majority_label(labels)

    # stop condition3: reach setted depth
    if max_depth is not None and depth >= max_depth:
        return majority_label(labels)

    

    best_feature = best_feature_threaded(data, thread_workers=thread_workers)
    root_name = names[best_feature]
    tree = {root_name: {}}

    child_feature_names = np.delete(names, best_feature)
    unique_values = np.unique(data[:, best_feature])
    for value in unique_values:
        child_data = split_dataset_np(data, best_feature, value)
        if child_data.shape[0] == 0:
            tree[root_name][int(value)] = majority_label(labels)
        else:
            tree[root_name][int(value)] = build_tree(
                child_data,
                child_feature_names,
                min_sample_size = min_sample_size,
                max_depth=max_depth,
                depth=depth + 1,
                thread_workers=thread_workers,
            )

    return tree


def predict_one(tree, feature_names, sample, default_label):
    node = tree
    names = np.asarray(feature_names, dtype=object)
    x = np.asarray(sample)

    while isinstance(node, dict):
        root = next(iter(node))
        children = node[root]

        matched = np.where(names == root)[0]
        if matched.size == 0:
            return default_label

        idx = int(matched[0])
        value = int(x[idx])
        if value not in children:
            return default_label

        node = children[value]
        names = np.delete(names, idx)
        x = np.delete(x, idx)

    return int(node)


def predict_batch(tree, feature_names, dataset, default_label):
    data = np.asarray(dataset)
    x = data[:, :-1]
    preds = np.empty(x.shape[0], dtype=np.int64)
    for i in range(x.shape[0]):
        preds[i] = predict_one(tree, feature_names, x[i], default_label)
    return preds


def accuracy_np(y_true, y_pred):
    true_arr = np.asarray(y_true, dtype=np.int64)
    pred_arr = np.asarray(y_pred, dtype=np.int64)
    if true_arr.size == 0:
        return 0.0
    return float(np.mean(true_arr == pred_arr))


def post_prune_tree_reduced_error(tree, train_data, valid_data, feature_names):
    if not isinstance(tree, dict):
        return tree

    train_np = np.asarray(train_data)
    valid_np = np.asarray(valid_data)
    names = np.asarray(feature_names, dtype=object)

    if train_np.shape[0] == 0:
        return tree

    root = next(iter(tree))
    children = tree[root]
    root_idx = int(np.where(names == root)[0][0])
    child_names = np.delete(names, root_idx)

    pruned_children = {}
    for edge_val, child in children.items():
        child_train = split_dataset_np(train_np, root_idx, edge_val)
        child_valid = split_dataset_np(valid_np, root_idx, edge_val)
        pruned_children[edge_val] = post_prune_tree_reduced_error(child, child_train, child_valid, child_names)

    pruned_tree = {root: pruned_children}

    if valid_np.shape[0] == 0:
        return pruned_tree

    default = majority_label(train_np[:, -1])
    subtree_pred = predict_batch(pruned_tree, names, valid_np, default)
    subtree_acc = accuracy_np(valid_np[:, -1], subtree_pred)

    leaf_pred = np.full(valid_np.shape[0], default, dtype=np.int64)
    leaf_acc = accuracy_np(valid_np[:, -1], leaf_pred)

    if leaf_acc >= subtree_acc:
        return int(default)

    return pruned_tree

In [7]:
def count_leaf_nodes(tree):
    if not isinstance(tree, dict):
        return 1
    root = next(iter(tree))
    return int(sum(count_leaf_nodes(child) for child in tree[root].values()))


def tree_depth(tree):
    if not isinstance(tree, dict):
        return 1
    root = next(iter(tree))
    child_depths = np.fromiter(
        (tree_depth(child) for child in tree[root].values()),
        dtype=np.int64,
    )
    return int(1 + child_depths.max(initial=0))


def save_markdown_report(results_array, output_path):
    rows = np.asarray(results_array, dtype=object)
    lines = np.empty(rows.shape[0] + 8, dtype=object)
    lines[0] = "# Fold Results"
    lines[1] = ""
    lines[2] = "| Fold | Unpruned Acc | Pruned Acc | Leaves | Depth |"
    lines[3] = "|---:|---:|---:|---:|---:|"

    unpruned = np.empty(rows.shape[0], dtype=np.float64)
    pruned = np.empty(rows.shape[0], dtype=np.float64)

    for i, item in enumerate(rows):
        unpruned[i] = float(item["unpruned_accuracy"])
        pruned[i] = float(item["pruned_accuracy"])
        lines[i + 4] = (
            f"| {item['fold']} | {unpruned[i]:.4f} | {pruned[i]:.4f} | "
            f"{item['leaf_count']} | {item['depth']} |"
        )

    lines[-4] = ""
    lines[-3] = f"- Mean unpruned accuracy: {unpruned.mean():.4f}"
    lines[-2] = f"- Mean pruned accuracy: {pruned.mean():.4f}"
    lines[-1] = ""

    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    output_file.write_text("\n".join(lines.tolist()), encoding="utf-8")
    print(f"Saved report: {output_file}")

# Datasets used in this notebook:
# 1) Letter Recognition
# 2) Adult
# 3) Mushroom
#
# Install once if needed:
# %pip install ucimlrepo

In [5]:
# UCI IDs: Letter=59, Adult=2, Mushroom=73
letter_df = fetch_ucirepo(id=59).data.original.copy()
adult_df = fetch_ucirepo(id=2).data.original.copy()
mushroom_df = fetch_ucirepo(id=73).data.original.copy()

print(letter_df.shape)
print("Letter columns:", letter_df.columns.to_numpy(dtype=object))
print(adult_df.shape)
print("Adult columns:", adult_df.columns.to_numpy(dtype=object))
print(mushroom_df.shape)
print("mushroom columns:", mushroom_df.columns.to_numpy(dtype=object))
display(letter_df.head(5))
display(adult_df.head(5))
display(mushroom_df.head(5))

(20000, 17)
Letter columns: ['lettr' 'x-box' 'y-box' 'width' 'high' 'onpix' 'x-bar' 'y-bar' 'x2bar'
 'y2bar' 'xybar' 'x2ybr' 'xy2br' 'x-ege' 'xegvy' 'y-ege' 'yegvx']
(48842, 15)
Adult columns: ['age' 'workclass' 'fnlwgt' 'education' 'education-num' 'marital-status'
 'occupation' 'relationship' 'race' 'sex' 'capital-gain' 'capital-loss'
 'hours-per-week' 'native-country' 'income']
(8124, 23)
mushroom columns: ['cap-shape' 'cap-surface' 'cap-color' 'bruises' 'odor' 'gill-attachment'
 'gill-spacing' 'gill-size' 'gill-color' 'stalk-shape' 'stalk-root'
 'stalk-surface-above-ring' 'stalk-surface-below-ring'
 'stalk-color-above-ring' 'stalk-color-below-ring' 'veil-type'
 'veil-color' 'ring-number' 'ring-type' 'spore-print-color' 'population'
 'habitat' 'poisonous']


,lettr,x-box,y-box,width,high,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx
0,T,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8
1,I,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10
2,D,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9
3,N,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8
4,G,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,poisonous
0,x,s,n,t,p,f,c,n,k,e,...,w,w,p,w,o,p,k,s,u,p
1,x,s,y,t,a,f,c,b,k,e,...,w,w,p,w,o,p,n,n,g,e
2,b,s,w,t,l,f,c,b,n,e,...,w,w,p,w,o,p,n,n,m,e
3,x,y,w,t,p,f,c,n,n,e,...,w,w,p,w,o,p,k,s,u,p
4,x,s,g,f,n,f,w,b,k,t,...,w,w,p,w,o,e,n,a,g,e


In [ ]:
for column in mushroom_df.columns.tolist():
    print(f"==={column}===")
    print(mushroom_df[column].value_counts(dropna=False))

In [63]:
def bin_numeric_features(df, target_col, n_bins=5, strategy="quantile"):
    binned_df = df.copy()

    for col_name in binned_df.columns:
        if col_name == target_col:
            continue

        series = binned_df[col_name]
        if not pd.api.types.is_numeric_dtype(series):
            continue

        non_null = series.dropna()
        if non_null.nunique() <= 1:
            continue

        if strategy == "quantile":
            # qcut can fail when many duplicated values exist; fallback to cut.
            try:
                binned = pd.qcut(series, q=n_bins, labels=False, duplicates="drop")
            except ValueError:
                binned = pd.cut(series, bins=n_bins, labels=False, include_lowest=True)
        elif strategy == "uniform":
            binned = pd.cut(series, bins=n_bins, labels=False, include_lowest=True)
        else:
            raise ValueError("strategy must be 'quantile' or 'uniform'")

        binned_df[col_name] = binned

    return binned_df


def encode_dataframe_to_int_numpy(df, target_col, random_state=42):
    cols = df.columns.to_numpy(dtype=object)
    working_df = df.copy()
    rng = np.random.default_rng(random_state)

    # Drop feature columns that are entirely missing.
    drop_cols = []
    for col_name in cols:
        if col_name == target_col:
            continue
        if working_df[col_name].isna().all():
            drop_cols.append(col_name)
    if drop_cols:
        working_df = working_df.drop(columns=drop_cols)
        cols = working_df.columns.to_numpy(dtype=object)

    # Impute missing values only in features by sampling from each feature's
    # empirical distribution in observed (non-missing) values.
    for col_name in cols:
        if col_name == target_col:
            continue
        series = working_df[col_name]
        missing_mask = series.isna()
        if not missing_mask.any():
            continue

        observed = series[~missing_mask]
        if observed.shape[0] == 0:
            # Entirely-missing columns are dropped above.
            continue

        value_probs = observed.value_counts(dropna=True, normalize=True)
        sampled_values = rng.choice(
            value_probs.index.to_numpy(dtype=object),
            size=int(missing_mask.sum()),
            p=value_probs.to_numpy(dtype=np.float64),
        )
        working_df.loc[missing_mask, col_name] = sampled_values

    encoded = np.empty((working_df.shape[0], working_df.shape[1]), dtype=np.int64)
    for col_idx, col_name in enumerate(cols):
        series = working_df[col_name]
        if pd.api.types.is_numeric_dtype(series):
            encoded[:, col_idx] = pd.to_numeric(series, errors="coerce").fillna(0).to_numpy(dtype=np.int64)
        else:
            codes, _ = pd.factorize(series.astype(str), sort=True)
            encoded[:, col_idx] = codes.astype(np.int64)

    target_idx = int(np.where(cols == target_col)[0][0])
    feature_indices = np.delete(np.arange(cols.shape[0], dtype=np.int64), target_idx)
    feature_names = cols[feature_indices]

    dataset = np.concatenate(
        [encoded[:, feature_indices], encoded[:, target_idx:target_idx + 1]],
        axis=1,
    )
    return dataset, feature_names


def create_kfold_datasets_multiprocess(
    dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1.0,
    random_state=42,
    process_workers=None,
):
    data = np.asarray(dataset, dtype=np.int64)
    y = data[:, -1]

    rng = np.random.default_rng(random_state)
    class_values = np.unique(y)

    sampled_indices = np.empty(0, dtype=np.int64)
    for class_value in class_values:
        class_idx = np.where(y == class_value)[0]
        class_idx = rng.permutation(class_idx)
        if sample_size_ratio >= 1.0:
            take_count = class_idx.shape[0]
        else:
            take_count = max(1, int(class_idx.shape[0] * sample_size_ratio))
        sampled_indices = np.concatenate([sampled_indices, class_idx[:take_count]])

    sampled_indices = rng.permutation(sampled_indices)
    sampled_data = data[sampled_indices]
    y_sampled = sampled_data[:, -1]

    # 1) Build one fixed stratified test split.
    fixed_test_indices = np.empty(0, dtype=np.int64)
    trainval_indices = np.empty(0, dtype=np.int64)
    sampled_all_idx = np.arange(sampled_data.shape[0], dtype=np.int64)

    for class_value in np.unique(y_sampled):
        class_idx = sampled_all_idx[y_sampled == class_value]
        class_idx = rng.permutation(class_idx)
        test_count = int(class_idx.shape[0] * test_ratio)
        test_count = max(1, test_count)
        test_count = min(test_count, class_idx.shape[0] - 1)
        fixed_test_indices = np.concatenate([fixed_test_indices, class_idx[:test_count]])
        trainval_indices = np.concatenate([trainval_indices, class_idx[test_count:]])

    fixed_test_indices = rng.permutation(fixed_test_indices)
    trainval_indices = rng.permutation(trainval_indices)
    fixed_test_data = sampled_data[fixed_test_indices]
    trainval_data = sampled_data[trainval_indices]
    y_trainval = trainval_data[:, -1]

    workers = process_workers
    if workers is None:
        workers = max(1, min(k, (os.cpu_count() or 1) - 1))

    task_args = np.empty(class_values.shape[0], dtype=object)
    for i, class_value in enumerate(class_values):
        class_local_idx = np.where(y_trainval == class_value)[0]
        task_args[i] = (int(class_value), class_local_idx, int(k), int(random_state + 1000 + i))

    with ProcessPoolExecutor(max_workers=workers) as executor:
        class_chunks = tuple(executor.map(build_class_folds_worker, task_args))

    fold_validation_indices = np.empty(k, dtype=object)
    for i in range(k):
        fold_validation_indices[i] = np.empty(0, dtype=np.int64)

    for _, chunks in class_chunks:
        for fold_idx in range(k):
            fold_validation_indices[fold_idx] = np.concatenate([fold_validation_indices[fold_idx], chunks[fold_idx]])

    for fold_idx in range(k):
        fold_validation_indices[fold_idx] = rng.permutation(fold_validation_indices[fold_idx])

    all_indices = np.arange(trainval_data.shape[0], dtype=np.int64)
    fold_records = np.empty(k, dtype=object)

    for fold_idx in range(k):
        val_idx = fold_validation_indices[fold_idx]
        train_mask = np.ones(trainval_data.shape[0], dtype=bool)
        train_mask[val_idx] = False
        train_idx = all_indices[train_mask]

        fold_records[fold_idx] = {
            "fold": fold_idx + 1,
            "train": trainval_data[train_idx],
            "validation": trainval_data[val_idx],
            "test": fixed_test_data,
        }

    return fold_records


letter_df_binned = bin_numeric_features(
    letter_df,
    target_col="lettr",
    n_bins=5,
    strategy="quantile",
)

letter_dataset, letter_dataset_feature_names = encode_dataframe_to_int_numpy(letter_df_binned, target_col="lettr")
adult_dataset, adult_dataset_feature_names = encode_dataframe_to_int_numpy(adult_df, target_col="income")
mushroom_dataset, mushroom_dataset_feature_names = encode_dataframe_to_int_numpy(mushroom_df, target_col="poisonous")

print("Applied binning to letter dataset: n_bins=8, strategy='quantile'")
print(letter_dataset_feature_names)
print(adult_dataset_feature_names)
print(mushroom_dataset_feature_names)

letter_fold_datasets = create_kfold_datasets_multiprocess(
    letter_dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

adult_fold_datasets = create_kfold_datasets_multiprocess(
    adult_dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

mushroom_fold_datasets = create_kfold_datasets_multiprocess(
    mushroom_dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

dataset_names = ["letter", "adult", "mushroom"]
original_datasets = [letter_dataset, adult_dataset, mushroom_dataset]
folded_datasets = [letter_fold_datasets, adult_fold_datasets, mushroom_fold_datasets]


for dataset_name, original_dataset, folded_dataset in zip(dataset_names, original_datasets, folded_datasets):
    target_categories = np.unique(original_dataset[:, -1])
    missing_count = int(np.isnan(original_dataset.astype(np.float64)).sum())
    print(f"dataset: {dataset_name}")
    print("dataset shape:", original_dataset.shape)
    print("target categories:", target_categories.tolist())
    print("missing values after preprocessing:", missing_count)
    print("has missing values:", missing_count > 0)
    print("dataset preview (first 5 rows):")
    print(original_dataset[:5])
    print("validation size:", folded_dataset[0]["validation"].shape[0])
    for fold in folded_dataset:
        print(
            f"fold {fold['fold']}: train={fold['train'].shape[0]}, "
            f"test={fold['test'].shape[0]}, validation={fold['validation'].shape[0]}"
        )
    print("\n")



Applied binning to letter dataset: n_bins=8, strategy='quantile'
['x-box' 'y-box' 'width' 'high' 'onpix' 'x-bar' 'y-bar' 'x2bar' 'y2bar'
 'xybar' 'x2ybr' 'xy2br' 'x-ege' 'xegvy' 'y-ege' 'yegvx']
['age' 'workclass' 'fnlwgt' 'education' 'education-num' 'marital-status'
 'occupation' 'relationship' 'race' 'sex' 'capital-gain' 'capital-loss'
 'hours-per-week' 'native-country']
['cap-shape' 'cap-surface' 'cap-color' 'bruises' 'odor' 'gill-attachment'
 'gill-spacing' 'gill-size' 'gill-color' 'stalk-shape' 'stalk-root'
 'stalk-surface-above-ring' 'stalk-surface-below-ring'
 'stalk-color-above-ring' 'stalk-color-below-ring' 'veil-type'
 'veil-color' 'ring-number' 'ring-type' 'spore-print-color' 'population'
 'habitat']
dataset: letter
dataset shape: (20000, 17)
target categories: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
missing values after preprocessing: 0
has missing values: False
dataset preview (first 5 rows):
[[ 0  2  0  1  0  2  4  0 

# Train and evaluate all cases 

In [64]:
import time

def _score_prepruning_fold(
    fold_record,
    feature_names,
    min_sample_size,
    max_depth,
    thread_workers=8,
):
    train_data = np.asarray(fold_record["train"], dtype=np.int64)
    test_data = np.asarray(fold_record["test"], dtype=np.int64)
    validation_data = np.asarray(fold_record["validation"], dtype=np.int64)

    default_label = majority_label(train_data[:, -1])
    tree = build_tree(
        train_data,
        np.asarray(feature_names, dtype=object),
        min_sample_size=int(min_sample_size),
        max_depth=max_depth,
        thread_workers=thread_workers,
    )

    val_pred = predict_batch(tree, feature_names, validation_data, default_label)
    val_acc = accuracy_np(validation_data[:, -1], val_pred)

    test_pred = predict_batch(tree, feature_names, test_data, default_label)
    test_acc = accuracy_np(test_data[:, -1], test_pred)

    return {
        "fold": fold_record,
        "validation_accuracy": float(val_acc),
        "test_accuracy": float(test_acc),
        "leaf_count": int(count_leaf_nodes(tree)),
        "depth": int(tree_depth(tree)),
    }


def run_prepruning_all_folds(
    fold_datasets,
    feature_names,
    min_sample_sizes,
    max_depths,
    thread_workers=8,
):
    t0 = time.perf_counter()
    grid = [(int(mss), mxd) for mss in min_sample_sizes for mxd in max_depths]
    grid_total = len(grid)
    if grid_total == 0:
        raise ValueError("min_sample_sizes and max_depths must be non-empty")

    best = {
        "min_sample_size": None,
        "max_depth": None,
        "mean_validation_accuracy": -1.0,
        "mean_leaf_count": float("inf"),
        "per_fold": [],
        "validation_accuracies": np.array([], dtype=np.float64),
        "test_accuracies": np.array([], dtype=np.float64),
    }

    for k, (mss, mxd) in enumerate(grid, start=1):
        per_fold = []
        val_accs = []
        test_accs = []
        leaf_counts = []
        for fold_record in fold_datasets:
            result = _score_prepruning_fold(
                fold_record,
                feature_names,
                min_sample_size=mss,
                max_depth=mxd,
                thread_workers=thread_workers,
            )
            per_fold.append(result)
            val_accs.append(result["validation_accuracy"])
            test_accs.append(result["test_accuracy"])
            leaf_counts.append(result["leaf_count"])

        val_accs = np.asarray(val_accs, dtype=np.float64)
        test_accs = np.asarray(test_accs, dtype=np.float64)
        mean_val = float(val_accs.mean()) if val_accs.size else 0.0
        mean_leaf = float(np.mean(leaf_counts)) if leaf_counts else float("inf")

        is_better = (
            mean_val > best["mean_validation_accuracy"]
            or (mean_val == best["mean_validation_accuracy"] and mean_leaf < best["mean_leaf_count"])
        )
        if is_better:
            best.update(
                min_sample_size=mss,
                max_depth=mxd,
                mean_validation_accuracy=mean_val,
                mean_leaf_count=mean_leaf,
                per_fold=per_fold,
                validation_accuracies=val_accs,
                test_accuracies=test_accs,
            )

        mean_test_cfg = float(test_accs.mean()) if test_accs.size else 0.0
        print(
            f"grid {k}/{grid_total} | mss={mss} max_depth={mxd} "
            f"| mean val={mean_val:.4f} mean test={mean_test_cfg:.4f} "
            f"| best mss={best['min_sample_size']} max_depth={best['max_depth']} "
            f"best mean val={best['mean_validation_accuracy']:.4f}",
            flush=True,
        )

    elapsed = time.perf_counter() - t0
    mean_test = float(best["test_accuracies"].mean()) if best["test_accuracies"].size else 0.0
    print(
        "grid search done in "
        f"{elapsed:.1f}s | best mss={best['min_sample_size']} "
        f"max_depth={best['max_depth']} | mean val={best['mean_validation_accuracy']:.4f} "
        f"mean test={mean_test:.4f}",
        flush=True,
    )

    test_accs = best["test_accuracies"]
    val_accs = best["validation_accuracies"]
    return {
        "best_min_sample_size": best["min_sample_size"],
        "best_max_depth": best["max_depth"],
        "per_fold": best["per_fold"],
        "mean_validation_accuracy_for_selection": best["mean_validation_accuracy"],
        "mean_test_accuracy": float(test_accs.mean()) if test_accs.size else 0.0,
        "std_test_accuracy": float(test_accs.std(ddof=1)) if test_accs.size > 1 else 0.0,
        "test_accuracies": test_accs,
        "validation_accuracies": val_accs,
    }

### Stump vs unpruned vs post-pruned

- **Stump:** `max_depth=1`
- **Unpruned:** `min_sample_size=1`, `max_depth=None`
- **Pruned:** pre-tuning, grid search on min_sample_size and max_depth

In [65]:
# helpers for stump vs unpruned vs pre-pruned (no post-pruning)

def _fit_and_score_case(
    train_data,
    eval_data,  # validation set
    feature_names,
    min_sample_size,
    max_depth,
    thread_workers=8,
):
    default_label = majority_label(train_data[:, -1])
    tree = build_tree(
        train_data,
        np.asarray(feature_names, dtype=object),
        min_sample_size=int(min_sample_size),
        max_depth=max_depth,
        thread_workers=thread_workers,
    )
    eval_pred = predict_batch(tree, feature_names, eval_data, default_label)
    eval_acc = accuracy_np(eval_data[:, -1], eval_pred)
    return {
        "tree": tree,
        "validation_accuracy": float(eval_acc),
        "leaf_count": int(count_leaf_nodes(tree)),
        "depth": int(tree_depth(tree)),
    }



In [ ]:
def max_unpruned_depth_across_folds(
    fold_datasets,
    feature_names,
    thread_workers=8,
):
    max_depth = 0
    for fold_record in fold_datasets:
        train_data = np.asarray(fold_record["train"], dtype=np.int64)
        tree = build_tree(
            train_data,
            np.asarray(feature_names, dtype=object),
            min_sample_size=1,
            max_depth=None,
            thread_workers=thread_workers,
        )
        depth = int(tree_depth(tree))
        if depth > max_depth:
            max_depth = depth
    return max_depth


def evaluate_three_models_one_fold_preselected(
    fold_record,
    feature_names,
    prepruned_choice,
    thread_workers=8,
):
    if prepruned_choice is None:
        raise ValueError("prepruned_choice is required (use run_prepruning_all_folds)")

    train_data = np.asarray(fold_record["train"], dtype=np.int64)
    test_data = np.asarray(fold_record["test"], dtype=np.int64)          # final unbiased metric
    val_data = np.asarray(fold_record["validation"], dtype=np.int64)     # used for validation report
    fold_id = int(fold_record["fold"])

    # 1) Stump: max_depth=1, report on validation
    stump = _fit_and_score_case(
        train_data=train_data,
        eval_data=val_data,
        feature_names=feature_names,
        min_sample_size=1,
        max_depth=0,
        thread_workers=thread_workers,
    )

    # 2) Unpruned: min_sample_size=1, max_depth=None, report on validation
    unpruned = _fit_and_score_case(
        train_data=train_data,
        eval_data=val_data,
        feature_names=feature_names,
        min_sample_size=1,
        max_depth=None,
        thread_workers=thread_workers,
    )

    # unbiased final test metrics for fixed-model baselines
    default_label = majority_label(train_data[:, -1])
    stump_test_pred = predict_batch(stump["tree"], feature_names, test_data, default_label)
    stump_test_acc = float(accuracy_np(test_data[:, -1], stump_test_pred))
    unpruned_test_pred = predict_batch(unpruned["tree"], feature_names, test_data, default_label)
    unpruned_test_acc = float(accuracy_np(test_data[:, -1], unpruned_test_pred))

    # 3) Pre-pruned: chosen by mean validation accuracy across folds
    pre_mss = int(prepruned_choice["min_sample_size"])
    pre_md = prepruned_choice["max_depth"]
    selection_val = float(prepruned_choice.get("mean_validation_accuracy", np.nan))

    pre_tree = build_tree(
        train_data,
        np.asarray(feature_names, dtype=object),
        min_sample_size=pre_mss,
        max_depth=pre_md,
        thread_workers=thread_workers,
    )
    pre_val_pred = predict_batch(pre_tree, feature_names, val_data, default_label)
    pre_val_acc = float(accuracy_np(val_data[:, -1], pre_val_pred))
    pre_test_pred = predict_batch(pre_tree, feature_names, test_data, default_label)
    pre_test_acc = float(accuracy_np(test_data[:, -1], pre_test_pred))

    return [
        {
            "fold": fold_id,
            "model": "stump",
            "validation_accuracy": stump["validation_accuracy"],
            "leaf_count": stump["leaf_count"],
            "depth": stump["depth"],
            "best_min_sample_size": 1,
            "best_max_depth": 1,
            "selection_validation_accuracy": np.nan,
            "test_accuracy": stump_test_acc,
        },
        {
            "fold": fold_id,
            "model": "unpruned",
            "validation_accuracy": unpruned["validation_accuracy"],
            "leaf_count": unpruned["leaf_count"],
            "depth": unpruned["depth"],
            "best_min_sample_size": 1,
            "best_max_depth": None,
            "selection_validation_accuracy": np.nan,
            "test_accuracy": unpruned_test_acc,
        },
        {
            "fold": fold_id,
            "model": "pre_pruned",
            "validation_accuracy": pre_val_acc,
            "leaf_count": int(count_leaf_nodes(pre_tree)),
            "depth": int(tree_depth(pre_tree)),
            "best_min_sample_size": pre_mss,
            "best_max_depth": pre_md,
            "selection_validation_accuracy": selection_val,
            "test_accuracy": pre_test_acc,
        },
    ]


In [69]:
# run all datasets and summarize

from IPython.display import display

# You can widen/narrow this grid depending on runtime.
# Keep None if you want "no depth cap" to be a candidate in pre-pruning.
min_sample_sizes = list(range(1, 6, 2))

configs = [
    ("letter", letter_fold_datasets, letter_dataset_feature_names),
    ("adult", adult_fold_datasets, adult_dataset_feature_names),
    ("mushroom", mushroom_fold_datasets, mushroom_dataset_feature_names),
]

all_rows = []

for dataset_name, folds, feature_names in configs:
    print(f"\n=== {dataset_name} ===")

    max_unpruned_depth = max_unpruned_depth_across_folds(
        folds,
        feature_names,
        thread_workers=8,
    )
    max_depths = list(range(0, max_unpruned_depth + 1))
    print("max depth: ", max_depths)

    pre_results = run_prepruning_all_folds(
        fold_datasets=folds,
        feature_names=feature_names,
        min_sample_sizes=min_sample_sizes,
        max_depths=max_depths,
        thread_workers=8,
    )
    prepruned_choice = {
        "min_sample_size": pre_results["best_min_sample_size"],
        "max_depth": pre_results["best_max_depth"],
        "mean_validation_accuracy": pre_results["mean_validation_accuracy_for_selection"],
    }

    for fold_record in folds:
        fold_rows = evaluate_three_models_one_fold_preselected(
            fold_record=fold_record,
            feature_names=feature_names,
            prepruned_choice=prepruned_choice,
            thread_workers=8,
        )
        for r in fold_rows:
            r["dataset"] = dataset_name
            all_rows.append(r)

results_df = pd.DataFrame(all_rows)
results_df["depth_aligned_to_max_depth"] = results_df["depth"] - 1


# Per-fold detailed results
display(results_df.sort_values(["dataset", "fold", "model"]).reset_index(drop=True))

# Mean/std comparison table
summary_df = (
    results_df
    .groupby(["dataset", "model"], as_index=False)
    .agg(
        mean_test_accuracy=("test_accuracy", "mean"),
        std_test_accuracy=("test_accuracy", "std"),
        mean_leaf_count=("leaf_count", "mean"),
        mean_depth=("depth_aligned_to_max_depth", "mean"),
    )
    .sort_values(["dataset", "mean_test_accuracy"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n=== Summary (mean +/- std test accuracy) ===")
display(summary_df)

# Optional: show pre-pruned chosen hyperparameters frequency
pre_choices = (
    results_df[results_df["model"] == "pre_pruned"]
    .groupby(["dataset", "best_min_sample_size", "best_max_depth"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["dataset", "count"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n=== Pre-pruned hyperparameter choices ===")
display(pre_choices)


=== letter ===
max depth:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
grid 1/54 | mss=1 max_depth=0 | mean val=0.0406 mean test=0.0408 | best mss=1 max_depth=0 best mean val=0.0406
grid 2/54 | mss=1 max_depth=1 | mean val=0.1370 mean test=0.1409 | best mss=1 max_depth=1 best mean val=0.1370
grid 3/54 | mss=1 max_depth=2 | mean val=0.2889 mean test=0.2882 | best mss=1 max_depth=2 best mean val=0.2889
grid 4/54 | mss=1 max_depth=3 | mean val=0.5027 mean test=0.5162 | best mss=1 max_depth=3 best mean val=0.5027
grid 5/54 | mss=1 max_depth=4 | mean val=0.6892 mean test=0.7078 | best mss=1 max_depth=4 best mean val=0.6892
grid 6/54 | mss=1 max_depth=5 | mean val=0.7843 mean test=0.7826 | best mss=1 max_depth=5 best mean val=0.7843
grid 7/54 | mss=1 max_depth=6 | mean val=0.8083 mean test=0.8093 | best mss=1 max_depth=6 best mean val=0.8083
grid 8/54 | mss=1 max_depth=7 | mean val=0.8101 mean test=0.8098 | best mss=1 max_depth=7 best mean val=0.8101
grid 9/54 | mss=1 max

,fold,model,validation_accuracy,leaf_count,depth,best_min_sample_size,best_max_depth,selection_validation_accuracy,test_accuracy,dataset,depth_aligned_to_max_depth
0,1,pre_pruned,0.505912,1,1,1,0.0,0.506108,0.506246,adult,0
1,1,stump,0.467940,24847,2,1,1.0,NaN,0.468974,adult,1
2,1,unpruned,0.465666,35392,15,1,NaN,NaN,0.472660,adult,14
3,2,pre_pruned,0.505912,1,1,1,0.0,0.506108,0.506246,adult,0
4,2,stump,0.474079,24840,2,1,1.0,NaN,0.470408,adult,1
...,...,...,...,...,...,...,...,...,...,...,...
85,9,stump,0.982192,9,2,1,1.0,NaN,0.980271,mushroom,1
86,9,unpruned,1.000000,24,5,1,NaN,NaN,1.000000,mushroom,4
87,10,pre_pruned,1.000000,24,5,1,4.0,1.000000,1.000000,mushroom,4
88,10,stump,0.990411,9,2,1,1.0,NaN,0.980271,mushroom,1



=== Summary (mean +/- std test accuracy) ===


,dataset,model,mean_test_accuracy,std_test_accuracy,mean_leaf_count,mean_depth
0,adult,pre_pruned,0.506246,0.000000,1.0,0.0
1,adult,unpruned,0.471636,0.002968,35348.6,14.0
2,adult,stump,0.469711,0.001106,24824.2,1.0
3,letter,pre_pruned,0.809763,0.006100,3795.7,7.0
4,letter,unpruned,0.807700,0.005740,4111.6,16.0
5,letter,stump,0.140916,0.000000,5.0,1.0
6,mushroom,pre_pruned,1.000000,0.000000,24.0,4.0
7,mushroom,unpruned,1.000000,0.000000,24.0,4.0
8,mushroom,stump,0.980271,0.000000,9.0,1.0



=== Pre-pruned hyperparameter choices ===


,dataset,best_min_sample_size,best_max_depth,count
0,adult,1,0.0,10
1,letter,1,7.0,10
2,mushroom,1,4.0,10
